In [0]:
%sql
-- ============================================================================
-- Tabela......: mart_safra_credito
-- Camada......: Gold (Data Product)
--
-- Objetivo
-- ----------------------------------------------------------------------------
-- Consolidar indicadores da carteira por safra de contratação, permitindo
-- acompanhar o comportamento financeiro e a qualidade das diferentes gerações
-- de contratos ao longo do tempo.
--
-- Perguntas de negócio
-- ----------------------------------------------------------------------------
-- • Qual safra originou maior volume financeiro?
-- • Quais safras apresentam melhor qualidade?
-- • Quais safras possuem maior inadimplência?
-- • Qual safra apresenta melhor índice de recebimento?
--
-- Grão
-- ----------------------------------------------------------------------------
-- 1 linha por safra de contratação
--
-- Fontes
-- ----------------------------------------------------------------------------
-- gold.fact_carteira_credito
-- gold.fact_parcela_credito
-- ============================================================================

CREATE OR REPLACE TABLE dev_credit_risk.gold.mart_safra_credito

USING DELTA

AS

-- ============================================================================
-- Indicadores da Carteira
-- ============================================================================

WITH carteira AS (

SELECT

    safra_contratacao,

    COUNT(*)                                AS qtd_contratos,

    SUM(valor_contratado)                   AS valor_contratado,

    AVG(valor_contratado)                   AS ticket_medio,

    AVG(prazo_meses)                        AS prazo_medio,

    AVG(taxa_juros)                         AS taxa_media_juros

FROM dev_credit_risk.gold.fact_carteira_credito

GROUP BY

    safra_contratacao

),

-- ============================================================================
-- Indicadores de Cobrança
-- ============================================================================

cobranca AS (

SELECT

    fc.safra_contratacao,

    SUM(fp.valor_parcela)                   AS valor_esperado,

    SUM(COALESCE(fp.valor_pago,0))          AS valor_recebido,

    SUM(fp.valor_em_aberto)                 AS saldo_em_aberto,

    ROUND(

        SUM(COALESCE(fp.valor_pago,0))
        /
        NULLIF(SUM(fp.valor_parcela),0)

    ,4)                                     AS indice_recebimento,

    ROUND(

        SUM(fp.flag_pagamento_atrasado)
        /
        COUNT(*)

    ,4)                                     AS tx_atraso,

    ROUND(

        SUM(fp.flag_pagamento_default_90)
        /
        COUNT(*)

    ,4)                                     AS tx_default,

    COUNT(

        DISTINCT CASE

            WHEN fp.flag_pagamento_default_90 = 1

            THEN fp.id_contrato

        END

    )                                       AS contratos_default

FROM dev_credit_risk.gold.fact_parcela_credito fp

INNER JOIN dev_credit_risk.gold.fact_carteira_credito fc

ON fp.id_contrato = fc.id_contrato

GROUP BY

    fc.safra_contratacao

)

-- ============================================================================
-- Resultado Final
-- ============================================================================

SELECT

    c.safra_contratacao,

    --------------------------------------------------------------------------
    -- Carteira
    --------------------------------------------------------------------------

    c.qtd_contratos,

    c.valor_contratado,

    ROUND(

        c.valor_contratado

        /

        SUM(c.valor_contratado) OVER()

    ,4)                                     AS participacao_carteira,

    c.ticket_medio,

    c.prazo_medio,

    c.taxa_media_juros,

    --------------------------------------------------------------------------
    -- Cobrança
    --------------------------------------------------------------------------

    b.valor_esperado,

    b.valor_recebido,

    b.indice_recebimento,

    b.saldo_em_aberto,

    --------------------------------------------------------------------------
    -- Qualidade da Carteira
    --------------------------------------------------------------------------

    b.contratos_default,

    b.tx_atraso,

    b.tx_default,

    --------------------------------------------------------------------------
    -- Auditoria
    --------------------------------------------------------------------------

    current_timestamp()                     AS dt_carga

FROM carteira c

LEFT JOIN cobranca b

ON c.safra_contratacao = b.safra_contratacao

ORDER BY

    c.safra_contratacao;

In [0]:
%sql
select * from dev_credit_risk.gold.mart_safra_credito